|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Tensor parallelism<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: two shardings, one collective<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import torch

torch.manual_seed(0)
D, H, RANKS = 256, 1024, 4
x  = torch.randn(8, D)
W1 = torch.randn(D, H)/D**0.5
W2 = torch.randn(H, D)/H**0.5
reference = torch.relu(x @ W1) @ W2
print('reference', tuple(reference.shape))

Shard an MLP across four ranks in two different ways. Both ways are correct.
Count what each one costs in conversation.

All of it runs on the CPU. Stage 20 does the same work with real collectives.
This notebook is about the design rule. You must get the rule correct before
the collectives exist.

# Exercise 1: column, then row

In [ ]:
def mlp_parallel(x, W1, W2, ranks):
  """Return (output, number_of_all_reduces).

  Split W1 one way and W2 the other way, so that you need one collective.
  Decide which way round before you write it. relu is elementwise, so a
  rank can finish its own hidden units alone.
  """
  W1s = 
  W2s = 
  partials = 
  return , 

out, collectives = mlp_parallel(x, W1, W2, RANKS)
print(f'max difference {(out-reference).abs().max().item():.2e}')
print(f'collectives per block: {collectives}')

# Exercise 2: the other way round

Also correct. Count the collectives.

In [ ]:
def mlp_wrong(x, W1, W2, ranks):
  """Now do it the other way round: row-parallel FIRST.

  Each rank now holds a partial sum of the hidden activations, and relu
  is not linear, so you cannot apply it to a partial sum. Count what
  that costs."""
  W1s = list(W1.chunk(ranks, dim=0))
  xs  = list(x.chunk(ranks, dim=1))
  h = 
  h = torch.relu(h)
  W2s = list(W2.chunk(ranks, dim=0))
  hs  = list(h.chunk(ranks, dim=1))
  out = 
  return out, 

out2, c2 = mlp_wrong(x, W1, W2, RANKS)
print(f'max difference {(out2-reference).abs().max().item():.2e}   (still correct)')
print(f'collectives per block: {c2}   <- twice the talking, same answer')

# Exercise 3: what a collective costs, with no clock

Each side of this comparison is bytes divided by a bandwidth, so the
milliseconds cancel. Write the ratio of collective time to compute time. You
then find that the layer count also cancels.

Three things remain: your model, your batch, and one number about the machine.
That number is how many times faster the memory is than the interconnect.

In [ ]:
def comm_over_compute(batch, d_model, ranks, bw_ratio, collectives=1):
  """Collective time divided by compute time. No clock in it.

  A ring all-reduce moves 2(R-1)/R times the tensor. The tensor is B x d
  in bf16. There are L layers and C collectives in each layer. The compute
  side is the weights, W = 24 d^2 L bytes, read at HBM speed by R ranks.

  Write the ratio and watch the layer count and the clock both cancel.
  """
  return 

BW_RATIO = 25.0        # PCIe node: HBM is about 25x the interconnect

print(f"{'batch':>6} {'1 collective':>14} {'2 collectives':>15}")
for b in (1, 32, 256):
  one = comm_over_compute(b, 4096, 8, BW_RATIO, 1)
  two = comm_over_compute(b, 4096, 8, BW_RATIO, 2)
  print(f'{b:>6} {one:>14.3f} {two:>15.3f}')
print('\n(ratio of talking to computing. 1.0 means half your step is the network.)')

# and the batch at which talking overtakes computing, for three machines
for ratio, name in ((1.0,'NVLink'), (25.0,'PCIe'), (200.0,'Ethernet')):
  
  print(f'{name:>9}: talking overtakes computing at batch {b1:>7,.0f} with 1 '
        f'collective, {b2:>7,.0f} with 2')

### Before you open the solution

1. Both versions give the right answer. Why does the second one need two
   collectives? Which operation is in the way?
2. State the rule in one sentence, so that it also tells you how to
   split attention.
3. Look at your table at batch 256. If you are on PCIe, what is the
   largest batch you can run before the collectives cost more than the
   step? What does that do to the throughput argument from Part 2?